# HR Database — Data Profiling & Regression Analysis

**Tables:** employees, departments, dept_emp, dept_manager, salaries, titles  
**Workflow:**
1. Load CSVs → SQLite `.db`
2. Profile each table
3. Build an analysis-ready flat table
4. Regression: salary ~ tenure, gender, department, title

## 0. Setup

In [ ]:
# Install deps if needed (Codespaces may already have these)
# !pip install pandas statsmodels matplotlib seaborn

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from pathlib import Path

DATA_DIR = Path('.')   # CSVs live next to this notebook
DB_PATH  = DATA_DIR / 'hr.db'

pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid')
print('Setup complete')

## 1. Load CSVs → SQLite

In [ ]:
TABLES = [
    'employees',
    'departments',
    'dept_emp',
    'dept_manager',
    'salaries',
    'titles',
]

conn = sqlite3.connect(DB_PATH)

for table in TABLES:
    df = pd.read_csv(DATA_DIR / f'{table}.csv')
    df.to_sql(table, conn, if_exists='replace', index=False)
    print(f'  {table:15s}  {len(df):>10,} rows  {list(df.columns)}')

print(f'\nDatabase saved → {DB_PATH.resolve()}')

## 2. Data Profiling

In [ ]:
def profile_table(name: str) -> pd.DataFrame:
    df = pd.read_sql(f'SELECT * FROM {name}', conn)
    stats = pd.DataFrame({
        'dtype':    df.dtypes,
        'nulls':    df.isna().sum(),
        'null_pct': (df.isna().mean() * 100).round(2),
        'unique':   df.nunique(),
        'sample':   [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })
    print(f'\n── {name.upper()} ── {len(df):,} rows × {len(df.columns)} cols')
    display(stats)
    return df

raw = {t: profile_table(t) for t in TABLES}

In [ ]:
# Date range check for time-series tables
for t in ['employees', 'salaries', 'titles', 'dept_emp']:
    df = raw[t]
    date_cols = [c for c in df.columns if 'date' in c]
    print(f'\n{t}')
    for c in date_cols:
        s = pd.to_datetime(df[c], errors='coerce')
        print(f'  {c}: {s.min().date()} → {s.max().date()}  ({s.isna().sum()} unparseable)')

In [ ]:
# Salary distribution
sal = raw['salaries'].copy()
print(sal['salary'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sal['salary'].plot.hist(bins=60, ax=axes[0], title='Salary Distribution (all records)')
sal['salary'].plot.box(ax=axes[1], title='Salary Box Plot')
plt.tight_layout()
plt.show()

In [ ]:
# Gender split
gender_counts = raw['employees']['gender'].value_counts()
print(gender_counts)
gender_counts.plot.pie(autopct='%1.1f%%', figsize=(4, 4), title='Gender Split')
plt.ylabel('')
plt.show()

In [ ]:
# Title distribution
title_counts = raw['titles']['title'].value_counts()
print(title_counts)
title_counts.plot.barh(figsize=(7, 4), title='Title Counts (all records incl. history)')
plt.tight_layout()
plt.show()

## 3. Build Analysis Table

One row per employee using **current** (most recent) salary, title, and department.

In [ ]:
query = """
WITH current_salary AS (
    SELECT emp_no, salary
    FROM salaries
    WHERE to_date = '9999-01-01'
),
current_title AS (
    SELECT emp_no, title
    FROM titles
    WHERE to_date = '9999-01-01'
),
current_dept AS (
    SELECT de.emp_no, de.dept_no, d.dept_name
    FROM dept_emp de
    JOIN departments d ON d.dept_no = de.dept_no
    WHERE de.to_date = '9999-01-01'
)
SELECT
    e.emp_no,
    e.gender,
    e.birth_date,
    e.hire_date,
    cd.dept_name,
    ct.title,
    cs.salary
FROM employees e
JOIN current_salary  cs ON cs.emp_no = e.emp_no
JOIN current_title   ct ON ct.emp_no = e.emp_no
JOIN current_dept    cd ON cd.emp_no = e.emp_no
"""

analysis = pd.read_sql(query, conn)
print(f'{len(analysis):,} active employees loaded')
analysis.head()

In [ ]:
# Feature engineering
ref_date = pd.Timestamp('2002-08-01')  # dataset end approx.

analysis['birth_date'] = pd.to_datetime(analysis['birth_date'])
analysis['hire_date']  = pd.to_datetime(analysis['hire_date'])

analysis['age_at_hire']   = ((analysis['hire_date'] - analysis['birth_date']).dt.days / 365.25).round(1)
analysis['tenure_years']  = ((ref_date - analysis['hire_date']).dt.days / 365.25).round(2)
analysis['gender_binary'] = (analysis['gender'] == 'M').astype(int)   # 1=M, 0=F

print(analysis[['salary', 'tenure_years', 'age_at_hire']].describe())

## 4. Exploratory Salary Analysis

In [ ]:
# Avg salary by department
dept_sal = analysis.groupby('dept_name')['salary'].mean().sort_values(ascending=False)
dept_sal.plot.barh(figsize=(8, 5), title='Average Current Salary by Department')
plt.xlabel('Salary ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Avg salary by title
title_sal = analysis.groupby('title')['salary'].mean().sort_values(ascending=False)
title_sal.plot.barh(figsize=(8, 5), title='Average Current Salary by Title')
plt.xlabel('Salary ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Salary by gender
analysis.groupby('gender')['salary'].describe()

In [ ]:
# Salary vs tenure scatter
fig, ax = plt.subplots(figsize=(9, 5))
for g, grp in analysis.groupby('gender'):
    ax.scatter(grp['tenure_years'], grp['salary'], alpha=0.15, s=5, label=g)
ax.set_xlabel('Tenure (years)')
ax.set_ylabel('Salary ($)')
ax.set_title('Salary vs Tenure by Gender')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Regression: Salary ~ Tenure + Gender + Department + Title

In [ ]:
# OLS — categorical vars handled via C() (dummy-coded, drops one level as reference)
model = smf.ols(
    'salary ~ tenure_years + age_at_hire + gender_binary + C(dept_name) + C(title)',
    data=analysis
).fit()

print(model.summary())

In [ ]:
# Extract coefficients for key predictors (non-category dummies)
coef = model.params.reset_index()
coef.columns = ['predictor', 'coef']
coef['abs_coef'] = coef['coef'].abs()
coef.sort_values('abs_coef', ascending=False).head(20)

In [ ]:
# Residual plot — check homoscedasticity
fitted   = model.fittedvalues
residuals = model.resid

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(fitted, residuals, alpha=0.1, s=4)
axes[0].axhline(0, color='red', lw=1)
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

pd.Series(residuals).plot.hist(bins=60, ax=axes[1], title='Residual Distribution')
plt.tight_layout()
plt.show()

print(f'R²: {model.rsquared:.4f}  |  Adj R²: {model.rsquared_adj:.4f}  |  N: {int(model.nobs):,}')

## 6. Save Analysis Table Back to SQLite

In [ ]:
analysis.to_sql('analysis_flat', conn, if_exists='replace', index=False)
print('Saved analysis_flat table to hr.db')

# Confirm all tables in db
tables_in_db = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables_in_db)

In [ ]:
conn.close()
print('Connection closed. hr.db is ready for SQL queries.')